## AWS Foundations: Services and the boto3 Pattern

# Developing with Core AWS Services: Foundations & Boto3

Welcome to the first lesson of our course, **Developing with Core AWS Services**.

Before we write any code, let's set the stage. In this lesson, you will learn what Amazon Web Services (AWS) is, get a quick tour of the core services used throughout the course, and learn how to use Python to communicate with them. This forms the foundation for everything we do—from storing files to managing databases.

---

## What Is AWS and Cloud Computing?

Traditionally, running an application required purchasing physical servers, setting them up on-premises, and maintaining power, cooling, and hardware. This model is expensive, slow to provision, and difficult to scale.

**Cloud computing** flips this model:

* Instead of owning hardware, you rent computing power, storage, and databases on demand from a cloud provider.
* You pay only for what you provision and consume.
* When workloads scale down, you release resources and cease paying.

**Amazon Web Services (AWS)** provides hundreds of modular building blocks to construct web platforms, mobile backends, data pipelines, and machine learning systems. In this course, you will control these building blocks directly via Python.

---

## Cloud Service Models: IaaS, PaaS, and SaaS

Cloud services are categorized by how much infrastructure the provider manages versus the user:

* **IaaS (Infrastructure as a Service):** You rent raw building blocks—virtual machines, networking, and raw storage. You are responsible for configuring operating systems, software runtimes, and application logic. *Example: Amazon EC2.*
* **PaaS (Platform as a Service):** The cloud provider manages the operating system, patching, and hardware scaling, allowing you to focus purely on application code and data. *Example: Amazon DynamoDB.*
* **SaaS (Software as a Service):** End-user applications delivered entirely over the web without direct infrastructure management (e.g., hosted email or browser-based document editors).

---

## Services Covered in This Course

* **Amazon S3 (Simple Storage Service):** Scalable object storage for arbitrary files, images, and backups organized into logical containers called **buckets**.
* **Amazon DynamoDB:** A managed, low-latency NoSQL database engineered for key-value and document data models.
* **Amazon SQS & SNS:** Managed messaging services. **SQS (Simple Queue Service)** provides decoupled, asynchronous message queues between services; **SNS (Simple Notification Service)** provides pub/sub broadcast notifications.
* **Amazon EC2 (Elastic Compute Cloud):** Resizable virtual machines (instances) rented on demand to run custom operating systems and application stacks.

---

## The Universal Boto3 Pattern

To interact with AWS in Python, we use the official SDK: `boto3`.

Regardless of the target AWS service, interactions follow a consistent four-step workflow:

1. **Import** the SDK.
2. **Initialize a client** for the target service.
3. **Invoke an API operation**.
4. **Parse the dictionary response**.

```python
import boto3

# 1 & 2: Initialize STS client
sts = boto3.client("sts")

# 3: Perform operation (Who am I?)
identity = sts.get_caller_identity()

# 4: Process the dictionary response
account = identity["Account"]
arn = identity["Arn"]

print(f"Account ID: {account}")
print(f"Identity ARN: {arn}")

```

**Output:**

```text
Account ID: 123456789012
Identity ARN: arn:aws:iam::123456789012:user/learner

```

> **Credential Handling:** `boto3` automatically resolves credentials from the execution environment (environment variables, IAM roles, or local AWS credential profiles), avoiding hardcoded keys in source files.

---

## Applying the Pattern to Compute: Amazon EC2

The exact same workflow applies when querying virtual compute resources:

```python
import boto3

# 1 & 2: Initialize EC2 client
ec2 = boto3.client("ec2")

# 3: Retrieve instance metadata
response = ec2.describe_instances()

# 4: Parse nested reservation/instance objects
reservations = response.get("Reservations", [])
instances = [i for r in reservations for i in r["Instances"]]

print(f"Found {len(instances)} EC2 instance(s):")
for instance in instances:
    print(f" • {instance['InstanceId']} ({instance['State']['Name']})")

```

**Output:**

```text
Found 2 EC2 instance(s):
 • i-0a1b2c3d4e5f67890 (running)
 • i-0987654321fedcba0 (stopped)

```

> **Note:** A fresh AWS account may return `Found 0 EC2 instance(s)`. An empty instance list confirms that API authentication and connectivity succeeded.

---

## Robust Error Handling

Network interruptions, expired credentials, or missing IAM permissions should be handled via `botocore.exceptions`:

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

try:
    ec2 = boto3.client("ec2")
    ec2.describe_instances()
    print("Connection successful!")
except NoCredentialsError:
    print("❌ No credentials found. Please check your setup.")
except ClientError as e:
    print(f"❌ AWS returned an error: {e.response['Error']['Message']}")

```

* `NoCredentialsError`: Raised when `boto3` cannot locate valid credentials in the runtime environment.
* `ClientError`: Standard exception raised when an AWS service rejects a request (e.g., `AccessDenied`, `InvalidParameterValue`).

---

## Summary

* **Cloud Economics:** Transition from high upfront hardware CapEx to on-demand utility OpEx.
* **Service Tiers:** Balance control vs. maintenance across IaaS, PaaS, and SaaS.
* **Consistency:** The `boto3.client('<service>')` paradigm provides a unified interface across all AWS APIs.
* **Resilience:** Wrap SDK invocations in structured `ClientError` / `NoCredentialsError` blocks for production safety.

## Initialize Your First AWS Client

Now that you have seen the boto3 Universal Pattern in the lesson, let's put its first step into practice.

You have been given a test_connection() function that is almost ready. It can already ask AWS, "Who am I?", and handle errors — but it is missing the lines that get everything started.

Your job is to fill in that initialization step:

    Import the boto3 library at the top of the file, where the first TODO is.
    Create an STS client inside the try block, at the second TODO, and store it in a variable named sts (the rest of the code expects that name).

💡 Notice that you never type a password or access key anywhere. boto3 finds your credentials automatically through its default credential provider chain, keeping your secrets out of your source code.

You'll extend this same function to a second service (EC2) in an upcoming exercise — for now, focus on getting that first connection working.

Add the two lines, run the file, and watch your first AWS connection succeed.

```
# TODO: Import the boto3 library
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        # TODO: Create an STS client and store it in a variable named `sts`

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```

Here is the completed code with `boto3` imported and the STS client initialized:

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client("sts")

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```

## Reading the Response from AWS

Nice work getting your STS client up and running in the previous exercise! With the connection ready, the next step of the Universal Pattern is to process the response that AWS sends back.

The test_connection() function already creates the client and calls sts.get_caller_identity(), storing the result in a variable named identity. Remember that identity is simply a Python dictionary, so you can read values directly from it using their keys.

Currently, account and arn are both set to None, which is why the program prints None for each. Your job is to replace those placeholders with real data:

    At the first TODO, read the "Account" key from identity and store it in account.
    At the second TODO, read the "Arn" key from identity and store it in arn.

The print statements are already written for you, so once you fill in these two lines, run the file and watch your real account ID and identity ARN appear on the screen. In the next exercise, you'll extend this same pattern to a brand-new service.

```
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client('sts')

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        # TODO: Read the "Account" key from identity and store it in account
        account = None
        # TODO: Read the "Arn" key from identity and store it in arn
        arn = None
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()
```

Here is the updated code reading the values directly from the `identity` dictionary:

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client("sts")

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        # Read the "Account" and "Arn" keys from the response dictionary
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```

## Transfer the Pattern to EC2

You can now create a client and read its response — the heart of the Universal Pattern. Let's see that same skill pay off with a brand-new service.

Right after the identity check, your test_connection() function will interact with Amazon EC2 using the exact steps you already know. Here is what you should add in Step 3, where the TODO comments are:

    At the first TODO, create an EC2 client and store it in a variable named ec2.
    At the second TODO, call ec2.describe_instances() and store the result in a variable named response.
    At the third TODO, read the "Reservations" list from response with .get("Reservations", []) and store it in a variable named reservations.

The line that flattens the reservations into a list of instances, the print line, and the loop that displays each instance are already written; once you add these three lines, run the file to see your instances listed.

💡 EC2 groups its instances inside "reservations," so the provided code flattens them into a single list for you. Don't worry if you see Found 0 EC2 instance(s) — a fresh account may have no servers yet, and the call still succeeding is exactly what proves your connection works.

Note that you are not importing anything new here — you simply repeat initialize a client → perform an operation → read the response. Finish this step, and you will have proof that one pattern unlocks every service ahead.


```
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client('sts')

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        # Step 3: Prove the pattern transfers — use a different service (EC2)
        # TODO: Create an EC2 client and store it in a variable named `ec2`
        # TODO: Call ec2.describe_instances() and store the result in a variable named `response`
        # TODO: Read the "Reservations" list from `response` (default to []) and store it in `reservations`
        instances = [i for r in reservations for i in r["Instances"]]
        print(f"\n🖥️ Found {len(instances)} EC2 instance(s):")
        for instance in instances:
            print(f"   • {instance['InstanceId']} ({instance['State']['Name']})")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()
```

Here is the completed code with Step 3 implemented for Amazon EC2:

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client('sts')

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        # Step 3: Prove the pattern transfers — use a different service (EC2)
        ec2 = boto3.client('ec2')
        response = ec2.describe_instances()
        reservations = response.get("Reservations", [])
        
        instances = [i for r in reservations for i in r["Instances"]]
        print(f"\n🖥️ Found {len(instances)} EC2 instance(s):")
        for instance in instances:
            print(f"   • {instance['InstanceId']} ({instance['State']['Name']})")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```

## Catching Errors Before They Crash

You have wired up the full Universal Pattern across two services — well done! There is one finishing touch left: making the code safe when something goes wrong.

Right now, the operations run with no protection, so a missing credential or a permission problem would crash the program with a long traceback. Let's catch those issues and report them clearly instead.

Wrap the operations in a try/except block:

    At the first TODO, add a try: line and move the existing operations (Steps 1 – 3 and return True) inside it, indenting them by one level.
    At the second TODO, add an except clause for NoCredentialsError that prints a friendly "No credentials found" message and returns False.
    At the third TODO, add an except clause for ClientError as e that prints the message from e.response['Error']['Message'] and returns False.

💡 Both error types are already imported for you, so you can focus on the control flow.

Add these handlers, and your connection test will be ready for the real world — calm and clear even when AWS pushes back.

```
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    # TODO: Add a `try:` line here, then indent Steps 1 – 3 and `return True`
    #       by one level so they run inside the try block.

    # Step 1: Import & initialize a client for AWS STS (Security Token Service)
    sts = boto3.client('sts')

    # Step 2: Make your first authenticated call — "Who am I?"
    identity = sts.get_caller_identity()
    account = identity["Account"]
    arn = identity["Arn"]
    print("✅ Connection successful!")
    print(f"   Account ID:   {account}")
    print(f"   Identity ARN: {arn}")

    # Step 3: Prove the pattern transfers — use a different service (EC2)
    ec2 = boto3.client('ec2')
    response = ec2.describe_instances()
    reservations = response.get("Reservations", [])
    instances = [i for r in reservations for i in r["Instances"]]
    print(f"\n🖥️ Found {len(instances)} EC2 instance(s):")
    for instance in instances:
        print(f"   • {instance['InstanceId']} ({instance['State']['Name']})")

    return True

    # Step 4: Handle the most common failure cases gracefully
    # TODO: Add an `except NoCredentialsError:` clause that prints a friendly
    #       "No credentials found" message and returns False.

    # TODO: Add an `except ClientError as e:` clause that prints
    #       e.response['Error']['Message'] and returns False.


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()
```

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client('sts')

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        # Step 3: Prove the pattern transfers — use a different service (EC2)
        ec2 = boto3.client('ec2')
        response = ec2.describe_instances()
        reservations = response.get("Reservations", [])
        instances = [i for r in reservations for i in r["Instances"]]
        print(f"\n🖥️ Found {len(instances)} EC2 instance(s):")
        for instance in instances:
            print(f"   • {instance['InstanceId']} ({instance['State']['Name']})")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```

## Build the Complete Connection Test

You have now practiced every step of the Universal Pattern on its own — initializing a client, reading a response, transferring the pattern to a new service, and catching errors. In this final exercise, you will bring all of those pieces together into one complete test_connection() function.

The surrounding scaffolding is ready for you: the imports, the function definition, the try: line, both except handlers, and the runner at the bottom. Your task is to fill in the body of the try block by following the TODO comments in order:

    Step 1: Create an STS client and store it in a variable named sts.
    Step 2: Call sts.get_caller_identity() into identity, read the "Account" and "Arn" keys into account and arn, and then print the success message with both values.
    Step 3: Create an EC2 client named ec2, call ec2.describe_instances() into response, read the "Reservations" list with .get("Reservations", []) into reservations, flatten them into a list of instances, and then print the count and loop through the list to print each instance.
    Final step: return True once every operation succeeds.

💡 Match the variable names in the TODO comments exactly — the except handlers and print statements already expect those names.

Finish the body and run the file; you will walk away with a reusable connection tester that you can drop into every project ahead.

```
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        # TODO: Create an STS client and store it in a variable named `sts`

        # Step 2: Make your first authenticated call — "Who am I?"
        # TODO: Call sts.get_caller_identity() and store the result in `identity`
        # TODO: Read the "Account" key from identity into a variable named `account`
        # TODO: Read the "Arn" key from identity into a variable named `arn`
        # TODO: Print "✅ Connection successful!" followed by the account ID and ARN

        # Step 3: Prove the pattern transfers — use a different service (EC2)
        # TODO: Create an EC2 client and store it in a variable named `ec2`
        # TODO: Call ec2.describe_instances() and store the result in a variable named `response`
        # TODO: Read the "Reservations" list from `response` (default to []) into `reservations`
        # TODO: Flatten the reservations into a list of instances called `instances`
        # TODO: Print how many instances you found, then loop through `instances` and print each one

        # TODO: Return True once every operation above succeeds

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()
```

```python
import boto3
from botocore.exceptions import ClientError, NoCredentialsError


def test_connection():
    """Verify AWS credentials and demonstrate the boto3 universal pattern."""
    try:
        # Step 1: Import & initialize a client for AWS STS (Security Token Service)
        sts = boto3.client("sts")

        # Step 2: Make your first authenticated call — "Who am I?"
        identity = sts.get_caller_identity()
        account = identity["Account"]
        arn = identity["Arn"]
        print("✅ Connection successful!")
        print(f"   Account ID:   {account}")
        print(f"   Identity ARN: {arn}")

        # Step 3: Prove the pattern transfers — use a different service (EC2)
        ec2 = boto3.client("ec2")
        response = ec2.describe_instances()
        reservations = response.get("Reservations", [])
        instances = [i for r in reservations for i in r["Instances"]]
        print(f"\n🖥️ Found {len(instances)} EC2 instance(s):")
        for instance in instances:
            print(f"   • {instance['InstanceId']} ({instance['State']['Name']})")

        return True

    # Step 4: Handle the most common failure cases gracefully
    except NoCredentialsError:
        print("❌ No credentials found. Check your environment configuration.")
        return False
    except ClientError as e:
        print(f"❌ AWS returned an error: {e.response['Error']['Message']}")
        return False


if __name__ == "__main__":
    # Step 5: The complete connection test, combining every building block
    print("🔐 AWS Connection Test (boto3 Universal Pattern)\n")
    test_connection()

```